# Analysis of data for the Essential FFPE Panel

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture

from tqdm import trange

### BatchDetect module imports

Import project-specific helpers from the `batchdetect` package:

- `load_thal_cross_lot_covs` and related loaders for Thalassemia data.
- `HeavyMixture` and `parametric_bootstrap_lrt` for mixture modeling and
  parametric bootstrap-based likelihood ratio tests.
- Correlation-based clustering utilities:
  `cluster_hierarchical_corr`, `cluster_spectral_corr`,
  `cluster_pca_kmeans_corr`.
- Correlation preprocessing helpers: `normalize_mat` and `get_correlations`.

These functions implement the main batch-detection and clustering methods used
throughout the analysis.


In [2]:
from batchdetect.loader import load_thal_cross_lot_covs
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt

In [3]:
df_thal_likelihoods = pd.read_csv('../likelihoods_thal131.csv')
print(df_thal_likelihoods.head())
snames = df_thal_likelihoods['Sample Name'].values
likelihoods = df_thal_likelihoods['Likelihood'].values
y_hat = df_thal_likelihoods['Label'].values

                      Sample Name  Likelihood  Label
0     RDvGMpTHALD2d200302iM1-PB05   39.361303      0
1     RDvGMpTHALD2d200302iM1-PB06   39.298358      0
2     RDvGMpTHALD2d200302iM1-PB07   66.229424      0
3      RDvGMpTHALD2d200302iM1-TV2   38.732541      0
4  RDvGMpTHALD2d200302iM1-aTHAL31 -214.132134      0


### Load  count data and metadata

Load the cross-lot coverage
matrix and associated metadata:

- `counts_thal`: amplicon-level coverage counts per sample.
- `y_thal`: sample labels from the loader (if provided).
- `sample_id`: sample identifiers.
- `features`: amplicon/feature annotations.

These raw counts are used for correlation-based clustering and neighborhood
analysis.


In [4]:
counts_thal,y_thal,sample_id,features,_ = load_thal_cross_lot_covs()
counts_thal = counts_thal[:-1]
y_thal = y_thal[:-1]
sample_id = sample_id[:-1]

In [5]:
plt.hist(likelihoods_new[y_new==0]);

NameError: name 'likelihoods_new' is not defined

In [ ]:
lp = likelihoods_new[y_new==0]
sp = snames_new[y_new==0]

In [ ]:
index_homozygous_del = [4,5,11,18,19,24,32,36,57]
counts_thal_new = np.delete(counts_thal, index_homozygous_del, axis=0)
likelihoods_new = np.delete(likelihoods, index_homozygous_del)
y_new = np.delete(y_hat, index_homozygous_del)
snames_new = np.delete(snames,index_homozygous_del)

In [ ]:
def null_factory():
    return HeavyMixture(
                n_components=1,
                component_distribution='gennorm',
                n_init=3,
                max_iter=1000,
            )   

def alt_factory():
            return HeavyMixture(
                n_components=2,
                component_distribution='gennorm',
                n_init=3,
                max_iter=1000,
            )   

In [ ]:
n_rep = 3
rng = np.random.default_rng(2021)
res_list = []
for i in trange(n_rep):
    idx = rng.choice(len(y_new),replace=True,size=len(y_new))
    X_sub = likelihoods_new[idx]
    y_sub = y_new[idx]
    res = parametric_bootstrap_lrt(
            X_sub,  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=500,
            random_state=i,
        )
    res_list.append(res)

In [ ]:
print([res['p_value'] for res in res_list])

In [ ]:
like_pos = likelihoods_new[y_new==0]
like_pos

In [ ]:
snames_pos = snames_new[y_new==0]
snames_pos[like_pos< -100]

In [ ]:
like_pos_sub = like_pos[like_pos>-100]

In [ ]:
snames_new[y_new==0]

In [ ]:
plt.hist(like_pos_sub,50);

In [ ]:
n_rep = 30
rng = np.random.default_rng(2021)
res_list_pos = []
for i in trange(n_rep):
    idx = rng.choice(len(like_pos_sub),replace=True,size=len(like_pos_sub))
    X_sub = like_pos_sub[idx]
    y_sub = y_new[idx]
    res = parametric_bootstrap_lrt(
            X_sub,  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=500,
            random_state=i,
        )
    res_list_pos.append(res)

In [ ]:
a = np.array([res['p_value'] for res in res_list_pos])
print(a)
print(np.mean(a))
print(np.std(a))